In [1]:
# %% [markdown]
# # # ETS-ANN Hybrid Model; forecast horizon = 1
# # # Python version 3.11+

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time  

# Data and Preprocessing
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split 

# ETS-ANN specific
from pycaret.time_series import TSForecastingExperiment, setup, create_model, compare_models, predict_model, get_config
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

# Visualization
import plotly.graph_objects as go
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# %% [markdown]
# ## 2. Configuration

# %%
# --- Defined Parameters ---
ticker = "XRP-USD" 
start_date = "2017-11-09"
end_date = "2025-01-01"

# Define Train/Test Split Ratio for the *initial* training phase
train_split_ratio = 0.80
# Define Validation Split Ratio *within* the initial training residuals for tuning
validation_split_ratio_for_tuning = 0.20 # 20% of initial residuals data for validation

# ANN Parameters
lags_ann = [1, 7, 30] 
max_lags = max(lags_ann)

# ANN Hyperparameter Tuning Grid
HP_ANN_NEURONS_OPTIONS = [25, 50, 100]
HP_ANN_EPOCHS_OPTIONS = [30, 50] 
HP_ANN_BATCH_SIZE_OPTIONS = [32, 64]
HP_ANN_LR_TUNE = 0.001

# Final ANN Training Configuration (if not tuned)
FINAL_ANN_EPOCHS = 50  
FINAL_ANN_BATCH_SIZE = 32 
FINAL_ANN_LR = 0.001 

RETRAIN_FREQUENCY = 0
# RETRAIN_EPOCHS_ANN = 5

# %% [markdown]
# ## 3. Data Loading and Preparation

# %%
print(f"--- Loading Data for {ticker} ---")
try:
    df_full = yf.download(tickers=[ticker], start=start_date, end=end_date, progress=False)
    if df_full.empty: raise ValueError(f"No data downloaded for {ticker}.")
    if 'Close' not in df_full.columns: raise ValueError(f"'Close' column not found.")
    df_full = df_full[['Close']].copy()
    df_full = df_full.asfreq('D')
    df_full.ffill(inplace=True); df_full.dropna(inplace=True)
    if df_full.empty: raise ValueError(f"Data became empty.")
    print(f"Loaded {len(df_full)} data points for {ticker} from {df_full.index.min()} to {df_full.index.max()}.")
except Exception as e:
    raise ValueError(f"Failed to load data for {ticker}: {e}")

# Split Data
n_total = len(df_full)
n_train = int(train_split_ratio * n_total)  
n_test = n_total - n_train 

train_data_df = df_full[:n_train]
test_data_df = df_full[n_train:]

print(f"\nInitial Training Data: {n_train} points ({train_data_df.index.min().strftime('%Y-%m-%d')} to {train_data_df.index.max().strftime('%Y-%m-%d')})")
print(f"Test Data (for walk-forward): {n_test} points ({test_data_df.index.min().strftime('%Y-%m-%d')} to {test_data_df.index.max().strftime('%Y-%m-%d')})")

# %% [markdown]
# ## 4. Initial ETS Model Selection and Fit

# %%
print("\n--- Initial ETS Model Training ---")
start_time_initial_ets = time.time()
exp_ets_initial = TSForecastingExperiment()
setup(data=train_data_df, fh=min(n_test, 30), session_id=123, verbose=False, numeric_imputation_target="ffill")
print("Comparing ETS models...")
best_ets_model_obj = compare_models(include=['ets', 'exp_smooth'], sort='RMSE', n_select=1, verbose=False)
print(f"Initial ETS Model Selected: {best_ets_model_obj}")
initial_ets_model_fitted = create_model(best_ets_model_obj, verbose=False)
end_time_initial_ets = time.time()
print(f"Initial ETS training finished in {end_time_initial_ets - start_time_initial_ets:.2f} seconds.")

# %% [markdown]
# ## 5. Initial Residual Calculation and Scaling

# %%
print("\n--- Calculating Initial Residuals ---")
y_train_pycaret = get_config('y_train')
try:
    ets_fitted_values_train = initial_ets_model_fitted._fitted_forecaster.fittedvalues
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()
except AttributeError:
    print("Warning: Fallback - Predicting ETS on initial train data.")
    ets_fitted_values_train = predict_model(initial_ets_model_fitted, data=y_train_pycaret)['y_pred']
    ets_fitted_values_train = ets_fitted_values_train.reindex(y_train_pycaret.index).dropna()

y_train_pycaret_aligned = y_train_pycaret.reindex(ets_fitted_values_train.index)
residuals_initial_train = y_train_pycaret_aligned - ets_fitted_values_train
residuals_initial_train.dropna(inplace=True)
print(f"Calculated {len(residuals_initial_train)} initial residuals.")

print("\n--- Scaling Initial Residuals ---")
scaler_residuals = MinMaxScaler(feature_range=(-1, 1))
normalized_residuals_initial_train = scaler_residuals.fit_transform(residuals_initial_train.values.reshape(-1, 1))
normalized_residuals_initial_train_series = pd.Series(normalized_residuals_initial_train.flatten(), index=residuals_initial_train.index)
print("Residual scaler fitted.")

# %% [markdown]
# ## 6. Prepare Lagged Residual Data for ANN

# %%
print("\n--- Preparing Lagged Residual Features ---")
def create_lagged_features_ann(series, lags):
    lagged_data = pd.DataFrame(index=series.index)
    lagged_data['target'] = series
    for lag in lags:
        lagged_data[f'lag_{lag}'] = series.shift(lag)
    lagged_data.dropna(inplace=True)
    return lagged_data

lagged_norm_resid_initial = create_lagged_features_ann(normalized_residuals_initial_train_series, lags_ann)
X_ann_initial_features = lagged_norm_resid_initial.drop('target', axis=1)
y_ann_initial_target = lagged_norm_resid_initial['target']
print(f"Created lagged residual dataset with shape: {X_ann_initial_features.shape}")

# %% [markdown]
# ## 7. ANN Hyperparameter Tuning (on Initial Residuals)

# %%
print("\n--- Starting ANN Hyperparameter Tuning ---")
start_time_tuning = time.time()

# Split the initial residual data for tuning
X_ann_train_tune, X_ann_val_tune, y_ann_train_tune, y_ann_val_tune = train_test_split(
    X_ann_initial_features, y_ann_initial_target,
    test_size=validation_split_ratio_for_tuning,
    shuffle=False  
)
print(f"ANN Tuning Train shape: {X_ann_train_tune.shape}")
print(f"ANN Tuning Validation shape: {X_ann_val_tune.shape}")

best_val_mse_tune = float('inf')
best_params_ann = None

for neurons in HP_ANN_NEURONS_OPTIONS:
    for epochs in HP_ANN_EPOCHS_OPTIONS:
        for batch_size in HP_ANN_BATCH_SIZE_OPTIONS:
            print(f"Tuning Trial: Neurons={neurons}, Epochs={epochs}, Batch Size={batch_size}")

            # Build ANN model for tuning
            ann_model_tune = Sequential([
                Dense(neurons, activation='relu', input_shape=(X_ann_train_tune.shape[1],)),
                Dense(max(10, neurons//2), activation='relu'),
                Dense(1) 
            ])
            ann_model_tune.compile(optimizer=Adam(learning_rate=HP_ANN_LR_TUNE), loss='mse')

            # Train model on tuning training set
            history = ann_model_tune.fit(X_ann_train_tune.values, y_ann_train_tune.values,
                                         epochs=epochs,
                                         batch_size=batch_size,
                                         validation_data=(X_ann_val_tune.values, y_ann_val_tune.values),
                                         verbose=0)  

            # Evaluate on tuning validation set
            if 'val_loss' in history.history and len(history.history['val_loss']) > 0:
                 val_mse = history.history['val_loss'][-1]
                 print(f"  Validation MSE: {val_mse:.6f}")
                 if val_mse < best_val_mse_tune:
                     best_val_mse_tune = val_mse
                     # Store the best parameters found so far
                     best_params_ann = {'neurons': neurons, 'epochs': epochs, 'batch_size': batch_size}
            else:
                 print("  Warning: No validation loss recorded.")

end_time_tuning = time.time()
print(f"\n--- ANN Tuning Complete in {end_time_tuning - start_time_tuning:.2f} seconds ---")
if best_params_ann is None:
     print("Warning: ANN Tuning failed to find best parameters. Using defaults.")
     # Use defaults defined in Configuration if tuning fails
     best_params_ann = {'neurons': HP_ANN_NEURONS, 'epochs': FINAL_ANN_EPOCHS, 'batch_size': FINAL_ANN_BATCH_SIZE}
else:
     print(f"Best Hyperparameters found: {best_params_ann}")
     print(f"Best Validation MSE during tuning: {best_val_mse_tune:.6f}")

# %% [markdown]
# ## 8. Train Final Initial ANN Model

# %%
print("\n--- Training Final Initial ANN Model on Residuals ---")
start_time_initial_ann = time.time()

# Build the final initial ANN model using the best found hyperparameters
final_ann_model = Sequential([
    Dense(best_params_ann['neurons'], activation='relu', input_shape=(X_ann_initial_features.shape[1],)),
    Dense(max(10, best_params_ann['neurons']//2), activation='relu'),
    Dense(1)
])
final_ann_model.compile(optimizer=Adam(learning_rate=FINAL_ANN_LR), loss='mse')

print(f"Training final initial ANN with {best_params_ann['neurons']} neurons for {best_params_ann['epochs']} epochs...")
# Train on the *entire* initial lagged residual dataset
final_ann_model.fit(X_ann_initial_features.values, y_ann_initial_target.values,
                    epochs=best_params_ann['epochs'], 
                    batch_size=best_params_ann['batch_size'],  
                    verbose=0) 

end_time_initial_ann = time.time()
print(f"Final Initial ANN training complete in {end_time_initial_ann - start_time_initial_ann:.2f} seconds.")
final_ann_model.summary()

# %% [markdown]
# ## 9. Prepare for Walk-Forward Loop

# %%
print("\n--- Preparing for Walk-Forward ---")
try:
    fitted_ets_forecaster = initial_ets_model_fitted._fitted_forecaster
    print(f"Using underlying forecaster: {type(fitted_ets_forecaster)}")
except AttributeError:
    raise AttributeError("Could not access the underlying '_fitted_forecaster' object.")

# Initialize history (starts with initial training residuals)
history_norm_residuals = normalized_residuals_initial_train_series.tolist()
if len(history_norm_residuals) < max_lags:
     print(f"Warning: Padding initial residual history.")
     history_norm_residuals = [0.0] * (max_lags - len(history_norm_residuals)) + history_norm_residuals
print(f"Initial residual history length: {len(history_norm_residuals)}")

# List to store final hybrid predictions (one per step)
ets_ann_walk_forward_predictions = []

# %% [markdown]
# ## 10. Walk-Forward Validation (Rolling Forecast) Loop - t+1

# %%
print(f"\n--- Starting ETS-ANN Walk-Forward Validation for {n_test} steps (t+1) ---")
start_time_walk_forward = time.time()

test_indices = test_data_df.index

for i, current_test_date in enumerate(test_indices):
    # Index in the FULL dataset for the point we are predicting
    prediction_target_index = n_train + i

    # --- ETS Prediction (Component 1: Č_{t+1}^1) ---
    try:
        # Predict the single next step
        ets_forecast_t_plus_1 = fitted_ets_forecaster.predict(start=prediction_target_index, end=prediction_target_index)[0]
    except Exception as e:
        print(f"Warning: ETS predict failed at step {i+1} for date {current_test_date}. Error: {e}. Using fallback.")
        if ets_ann_walk_forward_predictions:  
             prev_pred_idx = prediction_target_index - 1
             try: ets_forecast_t_plus_1 = fitted_ets_forecaster.predict(start=prev_pred_idx, end=prev_pred_idx)[0]
             except Exception: ets_forecast_t_plus_1 = ets_ann_walk_forward_predictions[-1]
        else: ets_forecast_t_plus_1 = train_data_df['Close'].iloc[-1]

    # --- ANN Prediction (Component 2: Č_{t+1}^2) ---
    ann_pred_t_plus_1_denorm = 0
    if len(history_norm_residuals) >= max_lags:
        current_input_features = [history_norm_residuals[-lag] for lag in lags_ann]
        input_vector = np.array(current_input_features).reshape(1, -1)
        # Use the tuned final_ann_model
        ann_pred_t_plus_1_norm = final_ann_model.predict(input_vector, verbose=0)[0, 0]
        ann_pred_t_plus_1_denorm = scaler_residuals.inverse_transform([[ann_pred_t_plus_1_norm]])[0, 0]
    else:
        print(f"Warning: Not enough residual history for ANN prediction at step {i+1}")

    # --- Combine Forecasts ---
    final_pred_t_plus_1 = ets_forecast_t_plus_1 + ann_pred_t_plus_1_denorm
    ets_ann_walk_forward_predictions.append(final_pred_t_plus_1)

    # --- Update History with Actual Values for the Current Test Date ---
    actual_price_t = test_data_df.loc[current_test_date, 'Close']

    # Get ETS forecast for the current step 't' to calculate actual residual
    current_actual_index = n_train + i - 1  
    try:
        ets_forecast_t = fitted_ets_forecaster.predict(start=current_actual_index, end=current_actual_index)[0]
    except Exception as e_ets_t:
        print(f"Warning calculating residual: ETS predict for step t failed ({e_ets_t}). Using fallback.")
        # Use the ETS component *predicted for step t* in the previous iteration
        if i == 0: ets_forecast_t = ets_fitted_values_train.iloc[-1]
        else: ets_forecast_t = ets_forecast_t_plus_1  


    actual_residual_t = actual_price_t - ets_forecast_t
    try:
        actual_residual_float = float(actual_residual_t)
        actual_residual_t_norm = scaler_residuals.transform([[actual_residual_float]])[0, 0]
    except Exception as e_transform:
        print(f"ERROR during residual transform update at step {i+1}: {e_transform}")
        actual_residual_t_norm = 0.0

    # Append actual normalized residual to main history
    history_norm_residuals.append(actual_residual_t_norm)

    if (i + 1) % 100 == 0:
        print(f"ETS-ANN Walk-Forward Step {i+1}/{n_test} complete.")


end_time_walk_forward = time.time()
total_walk_forward_time = end_time_walk_forward - start_time_walk_forward
print(f"\nETS-ANN Walk-Forward finished in {total_walk_forward_time:.2f} seconds.")

ets_ann_walk_forward_predictions = np.array(ets_ann_walk_forward_predictions)
print(f"Length of final predictions: {len(ets_ann_walk_forward_predictions)}")
print(f"Length of actual test data: {len(test_data_df)}")

# %% [markdown]
# ## 11. Evaluate Walk-Forward Performance

# %%
# Define evaluation metrics function 
def evaluate_forecast(y_true, y_pred, model_name):
    """Calculates and prints standard evaluation metrics."""
    y_true_flat = y_true.flatten(); y_pred_flat = y_pred.flatten()
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    mape = mean_absolute_percentage_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    try: r2 = r2_score(y_true_flat, y_pred_flat)
    except ValueError: r2 = np.nan
    print(f"\n--- {model_name} Walk-Forward (t+1) Evaluation Results ---")
    print(f"RMSE: {rmse:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4%}, R²: {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# Evaluate against the actual unscaled test data
y_test_actual = test_data_df['Close'].values
if len(y_test_actual) != len(ets_ann_walk_forward_predictions):
    raise ValueError(f"Length mismatch: Actual test data ({len(y_test_actual)}) vs Predictions ({len(ets_ann_walk_forward_predictions)})")

ets_ann_wf_results = evaluate_forecast(y_test_actual, ets_ann_walk_forward_predictions, f"ETS-ANN ({ticker})")

# %% [markdown]
# ## 12. Visualize Walk-Forward Results

# %%
print("\n--- Plotting Walk-Forward Forecasts ---")
results_df_wf = pd.DataFrame({
    'Actual': y_test_actual.flatten(),
    f'ETS-ANN (t+1)': ets_ann_walk_forward_predictions.flatten()
}, index=test_data_df.index)

fig = go.Figure()
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf['Actual'], mode='lines', name='Actual Price (Test)', line=dict(color='black')))
fig.add_trace(go.Scatter(x=results_df_wf.index, y=results_df_wf[f'ETS-ANN (t+1)'], mode='lines', name='ETS-ANN Walk-Forward (t+1)', line=dict(color='blue', dash='dot')))
fig.update_layout(
    title=f'ETS-ANN Walk-Forward (t+1) Forecast Comparison for {ticker} (Tuned ANN)',
    xaxis_title="Date", yaxis_title="Price (USD)", legend_title="Data/Model", template="plotly_white"
)
fig.show()




--- Loading Data for XRP-USD ---
YF.download() has changed argument auto_adjust default to True
Loaded 2610 data points for XRP-USD from 2017-11-09 00:00:00 to 2024-12-31 00:00:00.

Initial Training Data: 2088 points (2017-11-09 to 2023-07-28)
Test Data (for walk-forward): 522 points (2023-07-29 to 2024-12-31)

--- Initial ETS Model Training ---
Comparing ETS models...
Initial ETS Model Selected: ExponentialSmoothing(seasonal='mul', sp=15, trend='add')
Initial ETS training finished in 23.34 seconds.

--- Calculating Initial Residuals ---
Calculated 2058 initial residuals.

--- Scaling Initial Residuals ---
Residual scaler fitted.

--- Preparing Lagged Residual Features ---
Created lagged residual dataset with shape: (2028, 3)

--- Starting ANN Hyperparameter Tuning ---
ANN Tuning Train shape: (1622, 3)
ANN Tuning Validation shape: (406, 3)
Tuning Trial: Neurons=25, Epochs=30, Batch Size=32
  Validation MSE: 0.000806
Tuning Trial: Neurons=25, Epochs=30, Batch Size=64
  Validation MSE: 0

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_36 (Dense)                │ (None, 50)             │           200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,505 (17.60 KB)

 Trainable params: 1,501 (5.86 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 3,004 (11.74 KB)


--- Preparing for Walk-Forward ---
Using underlying forecaster: <class 'statsmodels.tsa.holtwinters.results.HoltWintersResultsWrapper'>
Initial residual history length: 2058

--- Starting ETS-ANN Walk-Forward Validation for 522 steps (t+1) ---
ETS-ANN Walk-Forward Step 100/522 complete.
ETS-ANN Walk-Forward Step 200/522 complete.
ETS-ANN Walk-Forward Step 300/522 complete.
ETS-ANN Walk-Forward Step 400/522 complete.
ETS-ANN Walk-Forward Step 500/522 complete.

ETS-ANN Walk-Forward finished in 58.88 seconds.
Length of final predictions: 522
Length of actual test data: 522

--- ETS-ANN (XRP-USD) Walk-Forward (t+1) Evaluation Results ---
RMSE: 0.4759, MAE: 0.2342, MAPE: 24.3366%, R²: -0.1551

--- Plotting Walk-Forward Forecasts ---
